<a href="https://colab.research.google.com/github/ederquesado/Spark/blob/main/3_Exercicio_SPARK_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode
from pyspark.sql.functions import split
from pyspark.ml.feature import RFormula
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [2]:
spark = SparkSession.builder.appName("SparkML").getOrCreate()

In [3]:
iris_data = spark.read.csv("/content/iris.csv",inferSchema=True, header=True)
iris_data.show(3)


+-----------+----------+-----------+----------+-----------+
|sepallength|sepalwidth|petallength|petalwidth|      class|
+-----------+----------+-----------+----------+-----------+
|        5.1|       3.5|        1.4|       0.2|Iris-setosa|
|        4.9|       3.0|        1.4|       0.2|Iris-setosa|
|        4.7|       3.2|        1.3|       0.2|Iris-setosa|
+-----------+----------+-----------+----------+-----------+
only showing top 3 rows


In [4]:
formula = RFormula(formula="class ~ . ",featuresCol="features", labelCol="label", handleInvalid="skip")


In [5]:
iris_trans = formula.fit(iris_data).transform(iris_data).select("features","label")
iris_trans.show(3)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|  0.0|
|[4.9,3.0,1.4,0.2]|  0.0|
|[4.7,3.2,1.3,0.2]|  0.0|
+-----------------+-----+
only showing top 3 rows


In [6]:
irisTreino, irisTeste = iris_trans.randomSplit([0.7,0.3])

In [7]:
nb = NaiveBayes(smoothing=2,   labelCol="label", featuresCol="features")

In [8]:
model = nb.fit(irisTreino)

In [9]:
previsao = model.transform(irisTeste)
previsao.show()


+-----------------+-----+--------------------+--------------------+----------+
|         features|label|       rawPrediction|         probability|prediction|
+-----------------+-----+--------------------+--------------------+----------+
|[4.3,3.0,1.1,0.1]|  0.0|[-9.8050996173459...|[0.76295578547525...|       0.0|
|[4.4,3.2,1.3,0.2]|  0.0|[-10.835957095024...|[0.73621195714351...|       0.0|
|[4.8,3.4,1.6,0.2]|  0.0|[-11.915341667683...|[0.73156562371558...|       0.0|
|[4.9,2.4,3.3,1.0]|  1.0|[-17.005788671486...|[0.13445732342655...|       1.0|
|[4.9,2.5,4.5,1.7]|  2.0|[-21.914922660151...|[0.02580502727656...|       2.0|
|[5.1,3.7,1.5,0.4]|  0.0|[-12.979577217005...|[0.74028191318623...|       0.0|
|[5.1,3.8,1.9,0.4]|  0.0|[-13.856429192909...|[0.69238446549587...|       0.0|
|[5.3,3.7,1.5,0.2]|  0.0|[-12.408283175761...|[0.79397715492932...|       0.0|
|[5.5,2.4,3.8,1.1]|  1.0|[-18.748759584650...|[0.09630636722957...|       1.0|
|[5.6,2.8,4.9,2.0]|  2.0|[-24.579764665657...|[0.017

In [10]:
#accuracy
avaliar = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="label",metricName="accuracy")
avaliar = avaliar.evaluate(previsao)
print(avaliar)

0.9142857142857143
